# 04 · Análise integrada e baseline de resolução

**Desafio 1 — BRACIS 2026 / Jusbrasil**

Os três notebooks anteriores olharam cada base isoladamente. Este liga as três e
responde à pergunta operacional: **dado um trecho citado, dá para achar o registro
certo na base congelada?**

O caminho aqui é incremental — um baseline ingênuo, depois os consertos que os dados
pedem, medindo o ganho de cada um. O ponto de chegada não é uma solução, é o mapa de
onde a dificuldade realmente está.

In [ ]:
import sys, re, collections
from pathlib import Path

for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "src" / "eda_utils.py").exists():
        sys.path.insert(0, str(base / "src"))
        break

import pandas as pd
import matplotlib.pyplot as plt
import eda_utils as u

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("raiz do projeto:", u.raiz_projeto())

In [ ]:
gab = u.carregar_gabarito()
docs = u.carregar_documentos()
con = u.conectar_base()
df = pd.DataFrame(gab).drop(columns=["trecho_bruto"])
print(f"gabarito: {len(df)} citações · base: "
      f"{con.execute('SELECT COUNT(*) FROM documentos').fetchone()[0]} registros · "
      f"documentos: {len(docs)}")

## 1. O gabarito aponta para dentro da base?

In [ ]:
ids = sorted({int(r["id_canonico"]) for r in gab if r["id_canonico"].strip()})
achados = pd.read_sql_query(
    "SELECT id, documento_id, natureza, tipo, tribunal FROM documentos WHERE id IN (%s)"
    % ",".join(map(str, ids)), con)

print(f"ids distintos citados: {len(ids)} · encontrados na base: {len(achados)}")
print("\nnatureza dos alvos:")
print(achados.groupby(["natureza", "tipo"]).size().to_string())
print(f"\ncobertura: {len(ids)}/1016 registros = {100*len(ids)/1016:.1f}% da base é alvo de alguma citação")

In [ ]:
alvos = achados.groupby("tribunal").size().rename("citados")
acervo = pd.read_sql_query(
    "SELECT tribunal, COUNT(*) AS registros FROM documentos WHERE tribunal IS NOT NULL GROUP BY 1", con
).set_index("tribunal")["registros"]
comp = pd.concat([acervo, alvos], axis=1).fillna(0).astype(int)
comp["% citado"] = (100 * comp.citados / comp.registros).round(1)
print(comp.to_string())

comp[["registros", "citados"]].plot(kind="bar", color=["#B0B0B0", "#4C72B0"])
plt.ylabel("registros"); plt.title("A base é quase toda palheiro: 9,4% é alvo de citação")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

Todos os 95 identificadores do gabarito existem na base — a promessa de que "toda
citação `real` resolve dentro da cobertura" se confirma. Mas a distribuição é muito
desigual: o STJ tem 47 citações para 202 registros, enquanto o STF tem 3 para 201.
Os outros 921 registros existem para gerar candidatos falsos.

E os 18 registros normativos são **todos** alvo: as 5 súmulas e os 13 dispositivos
aparecem citados. Nenhum deles é palheiro.

## 2. Baseline v1 — o caminho ingênuo

Extrai os dígitos, reagrupa de três em três, busca por frase no FTS, e filtra por
posição com o limiar único que a documentação sugere.

In [ ]:
reais_jur = [r for r in gab if r["classificacao"] == "real" and r["tipo"] == "jurisprudencia"]
print(f"citações reais de jurisprudência: {len(reais_jur)}")

def baseline_v1(trecho, janela=300):
    digitos = re.sub(r"[^0-9]", "", re.search(r"\d[\d.\s]{4,}\d", trecho).group(0)) \
        if re.search(r"\d[\d.\s]{4,}\d", trecho) else ""
    if not digitos:
        return {}
    forma = u.agrupar_milhares(digitos)
    encontrados = {}
    try:
        for doc_id, texto in con.execute(
                "SELECT d.id, d.texto FROM documentos_fts f JOIN documentos d ON d.rowid=f.rowid "
                "WHERE documentos_fts MATCH ?", ('"' + forma + '"',)):
            pos = texto.find(forma)
            if 0 <= pos < janela:
                encontrados[doc_id] = pos
    except Exception:
        pass
    return encontrados

def avaliar(resolvedor, casos, rotulo):
    exatas = ambiguas = zero = erradas = 0
    falhas = []
    for r in casos:
        cand = resolvedor(r["trecho"])
        esperado = int(r["id_canonico"])
        if len(cand) == 1 and esperado in cand:
            exatas += 1
        elif not cand:
            zero += 1; falhas.append(("zero candidatos", r["trecho"]))
        elif esperado in cand:
            ambiguas += 1; falhas.append(("ambígua", r["trecho"]))
        else:
            erradas += 1; falhas.append(("candidato errado", r["trecho"]))
    print(f"{rotulo}: exatas {exatas}/{len(casos)} ({100*exatas/len(casos):.0f}%) · "
          f"ambíguas {ambiguas} · zero {zero} · erradas {erradas}")
    return exatas, falhas

_, falhas_v1 = avaliar(baseline_v1, reais_jur, "v1 (dígitos + FTS + janela 300)")

In [ ]:
print("por que falha:")
for motivo, trecho in falhas_v1[:16]:
    print(f"  [{motivo:<16}] {re.sub(chr(10), ' ', trecho)[:56]}")

Um terço acerta. As falhas se agrupam em famílias bem definidas, e cada família tem
um conserto específico — é isso que as próximas seções fazem.

## 3. Baseline v2 — as três correções que os dados pedem

1. **Número em formato CNJ** não pode ser reagrupado de três em três; e TST/TSE
   escrevem a parte sequencial sem os zeros à esquerda (`ARR-213-85.2010...`), então
   vale tentar as duas formas.
2. **Letra no lugar de dígito** (`21737l8`, `1.528.4S5`) precisa ser corrigida — mas
   **só dentro do token numérico**. Aplicar o mapa ao trecho inteiro transforma
   `RESP` em `RE5P` e destrói a sigla.
3. **A janela de cabeçalho é por tribunal**, não global (ver notebook 02): o TST só
   nomeia o processo perto do caractere 1.000.

Tudo isso está em `u.variantes_numero()` e `u.resolver_por_numero()`.

In [ ]:
for exemplo in ["AgInt no RESP 21737l8 - SP", "AgRg no RESP 1.528.4S5/ RJ",
                "R.Esp. n. 1.45g.779-MA", "ARR-213-85.2010.5.02.0030",
                "AREsp nº 1.996.496/RJ", "Súmula 331 do TST"]:
    print(f"  {exemplo:<32} -> {u.variantes_numero(exemplo)}")

In [ ]:
_, falhas_v2 = avaliar(u.resolver_por_numero, reais_jur, "v2 (variantes + janela por tribunal)")
print()
for motivo, trecho in falhas_v2:
    print(f"  [{motivo:<16}] {re.sub(chr(10), ' ', trecho)[:56]}")

De 35% para 89%. E as falhas que sobram têm quase todas a mesma cara: **súmula**.

## 4. O erro caro: súmula no resolvedor numérico

`Súmula 979 do STF` vira a busca pelo token `979`, que casa com qualquer acórdão que
tenha esse inteiro no cabeçalho. Como as súmulas da base não contêm o próprio número
(notebook 02), o resolvedor numérico nunca acerta uma súmula — mas frequentemente
"acerta" um acórdão errado.

Isso é exatamente o erro que a métrica pune com o fator τ: alucinação carimbada como
verificada.

In [ ]:
inventadas_jur = [r for r in gab if r["classificacao"] == "inventada" and r["tipo"] == "jurisprudencia"]

vazadas = [r for r in inventadas_jur if u.resolver_por_numero(r["trecho"])]
print(f"inventadas que o resolvedor numérico transformaria em 'real': {len(vazadas)}/{len(inventadas_jur)}")
for r in vazadas:
    print(f'  {re.sub(chr(10), " ", r["trecho"])[:44]:<44} -> variantes {u.variantes_numero(r["trecho"])}')

In [ ]:
# O roteador precisa tolerar OCR na própria palavra: "5úmula 211" tem S trocado por 5.
E_SUMULA = re.compile(r"[s5$][uúü][mn]|verbete", re.I)

capturadas = [r for r in gab if E_SUMULA.search(r["trecho"])]
print(f"menções de súmula capturadas pelo roteador: {len(capturadas)}")
print(pd.Series([r["classificacao"] for r in capturadas]).value_counts().to_string())

def resolver_v3(trecho):
    if E_SUMULA.search(trecho):
        return {}          # súmula não passa pelo caminho numérico
    return u.resolver_por_numero(trecho)

nao_sumula = [r for r in reais_jur if not E_SUMULA.search(r["trecho"])]
avaliar(resolver_v3, nao_sumula, "v3 (com roteamento de súmula)")

tau = sum(1 for r in inventadas_jur if resolver_v3(r["trecho"]))
print(f"\nvazamento τ (inventada -> real): {tau}/{len(inventadas_jur)} = {tau/len(inventadas_jur):.3f}")

Uma regra de roteamento de uma linha leva a acurácia de 89% para **95%** nas reais de
jurisprudência e derruba o vazamento τ para **2%**. O único vazamento remanescente é
`Reclamação n. 44- .921 (PE)`, onde o número mutilado faz o extrator ler só `921` — um
piso mínimo de dígitos resolve.

As cinco súmulas do gabarito se resolvem por uma tabela de cinco linhas, casando
número e tribunal. Não há caminho automático aqui, e nem precisa haver.

## 5. Trade-off: janela de cabeçalho × alucinação

A janela que recupera o TST é a mesma que aumenta a chance de confundir "o documento
que É o processo" com "um documento que o cita". Vale medir a curva.

In [ ]:
original = dict(u.JANELA_CABECALHO)
linhas = []
for j in [300, 500, 1000, 1500, 2000, 2500, 5000]:
    u.JANELA_CABECALHO.update({k: j for k in u.JANELA_CABECALHO})   # janela única, para comparar
    exatas = sum(1 for r in nao_sumula
                 if len(u.resolver_por_numero(r["trecho"])) == 1
                 and int(r["id_canonico"]) in u.resolver_por_numero(r["trecho"]))
    vaz = sum(1 for r in inventadas_jur
              if not E_SUMULA.search(r["trecho"]) and u.resolver_por_numero(r["trecho"]))
    linhas.append({"janela única": j, "acurácia %": round(100*exatas/len(nao_sumula), 1),
                   "τ (vazamento)": round(vaz/len(inventadas_jur), 3)})
u.JANELA_CABECALHO.update(original)   # restaura a configuração por tribunal

curva = pd.DataFrame(linhas)
print(curva.to_string(index=False))

fig, ax1 = plt.subplots()
ax1.plot(curva["janela única"], curva["acurácia %"], "o-", color="#4C72B0", label="acurácia")
ax1.set_xlabel("janela de cabeçalho (caracteres)"); ax1.set_ylabel("acurácia %", color="#4C72B0")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(curva["janela única"], curva["τ (vazamento)"], "s--", color="#C44E52", label="τ")
ax2.set_ylabel("τ — inventadas viradas real", color="#C44E52")
plt.title("Recuperar o TST custa exposição a alucinação")
plt.tight_layout(); plt.show()

In [ ]:
# E a configuração por tribunal, que é o padrão de eda_utils?
exatas = sum(1 for r in nao_sumula
             if len(u.resolver_por_numero(r["trecho"])) == 1
             and int(r["id_canonico"]) in u.resolver_por_numero(r["trecho"]))
vaz = sum(1 for r in inventadas_jur
          if not E_SUMULA.search(r["trecho"]) and u.resolver_por_numero(r["trecho"]))
print(f"janela por tribunal: acurácia {100*exatas/len(nao_sumula):.1f}% "
      f"({exatas}/{len(nao_sumula)}) · τ {vaz/len(inventadas_jur):.3f}")

A curva mostra o dilema com janela única: 300 caracteres dá τ baixo (0,02) mas perde o
TST inteiro (85,7%); abrir para 1.500 recupera o TST (93,5%) e dobra o vazamento
(0,04); passar de 2.500 começa a perder acurácia de novo, porque a janela larga faz
acórdãos que apenas citam entrarem como candidatos e a resolução vira ambígua.

**A configuração por tribunal domina as duas pontas**: 94,8% de acurácia com τ = 0,02 —
melhor acurácia que qualquer janela única *e* o menor vazamento. Não é um meio-termo,
é o resultado de usar a informação certa (cada tribunal formata seu cabeçalho do seu
jeito) em vez de um parâmetro global ajustado no olho.

O que a métrica cobra: `s = macroF1 · (1 − 0,5·τ)`. Com τ = 0,02 o custo é 1% da nota;
sem o roteamento de súmula (τ = 0,14) seriam 7%. Apertar o roteamento rende mais que
perseguir o último ponto de recall.

## 6. A classe `lei` resolve por tabela

Treze dispositivos na base, nenhum com colisão de número de artigo. Mas o número
sozinho não basta — e o contraexemplo é instrutivo.

In [ ]:
dispositivos = {}
for doc_id, texto in con.execute("SELECT id, texto FROM documentos WHERE natureza='dispositivo'"):
    # o texto abre com um título ("Artigo 5º da Constituição…"); o "Art. N." vem na linha seguinte
    m = re.search(r"^\s*Art\.?\s*([\d.]+)", texto, re.M)
    dispositivos[int(m.group(1).replace(".", ""))] = doc_id
print("artigo -> id canônico:", dispositivos)

ARTIGO = re.compile(r"art(?:igo)?\.?\s*([\d.]+)", re.I)
leis = [r for r in gab if r["tipo"] == "lei"]
acertos = 0
for r in leis:
    m = ARTIGO.search(re.sub(r"\s+", " ", r["trecho"]))
    if not m:
        previsto, alvo = "incompleta", None
    else:
        numero = int(m.group(1).replace(".", ""))
        alvo = dispositivos.get(numero)
        previsto = "real" if alvo else "inventada"
    ok = previsto == r["classificacao"] and (previsto != "real" or str(alvo) == r["id_canonico"])
    acertos += ok
    if not ok:
        print(f"  ERRO: {re.sub(chr(10), ' ', r['trecho'])} -> previsto {previsto}/{alvo}, "
              f"gabarito {r['classificacao']}/{r['id_canonico']}")
print(f"\nsó pelo número do artigo: {acertos}/{len(leis)}")

In [ ]:
# O conserto: rotular à mão o estatuto de cada um dos 13 dispositivos.
ESTATUTO = {
    10577194: "Código Eleitoral", 10590194: "Código Penal Militar",
    10606184: "Código de Defesa do Consumidor", 10626510: "Constituição",
    10637358: "CLT", 10641213: "Constituição", 10641516: "Constituição",
    10647746: "CLT", 10652044: "Código de Processo Penal", 10710324: "CLT",
    10718759: "Código Civil", 11304039: "Lei Complementar nº 64/1990",
    28893055: "Código de Processo Civil",
}
APELIDOS = {
    "Constituição": ["constitui", "cf/88", "carta magna"],
    "CLT": ["clt", "consolidação das leis do trabalho"],
    "Código de Processo Civil": ["cpc", "processo civil", "13.105"],
    "Código de Processo Penal": ["cpp", "processo penal"],
    "Código Penal Militar": ["cpm", "penal militar"],
    "Código Civil": ["código civil"],
    "Código de Defesa do Consumidor": ["cdc", "defesa do consumidor"],
    "Código Eleitoral": ["código eleitoral"],
    "Lei Complementar nº 64/1990": ["complementar n", "lc 64", "64/1990"],
}

def estatuto_citado(trecho):
    t = trecho.lower()
    for nome, chaves in APELIDOS.items():
        if any(k in t for k in chaves):
            return nome
    return None

acertos = 0
for r in leis:
    m = ARTIGO.search(re.sub(r"\s+", " ", r["trecho"]))
    if not m:
        previsto, alvo = "incompleta", None
    else:
        numero = int(m.group(1).replace(".", ""))
        alvo = dispositivos.get(numero)
        if alvo and ESTATUTO[alvo] == estatuto_citado(r["trecho"]):
            previsto = "real"
        else:
            previsto, alvo = "inventada", None
    ok = previsto == r["classificacao"] and (previsto != "real" or str(alvo) == r["id_canonico"])
    acertos += ok
    if not ok:
        print(f"  ERRO: {re.sub(chr(10), ' ', r['trecho'])} -> {previsto}/{alvo}")
print(f"\nnúmero do artigo + estatuto: {acertos}/{len(leis)}")

O caso que separa 29/30 de 30/30 é `art 290 da Constituição Federal`, marcado como
`inventada`. O artigo 290 **existe** na base — mas no Código Penal Militar. Casar só
pelo número resolve para o dispositivo errado e carimba uma alucinação como `real`:
de novo, o erro que a métrica pune com τ.

Rotular à mão os 13 dispositivos com seu estatuto custa quinze minutos e fecha a
classe `lei` em 100% na amostra de desenvolvimento.

## 7. As incompletas "buscáveis"

O terceiro subtipo: informação suficiente para consultar, insuficiente para
identificar um único registro.

In [ ]:
padrao = re.compile(r"\b(STF|STJ|TSE|TST|STM)\b.*?(\d{4})", re.S)
linhas = []
for r in gab:
    if r["classificacao"] != "incompleta":
        continue
    t = re.sub(r"\s+", " ", r["trecho"])
    m = padrao.search(t)
    if not m:
        linhas.append({"trecho": t[:56], "candidatos": None}); continue
    sobrenome = t.split()[-1]
    n = con.execute(
        "SELECT COUNT(*) FROM documentos WHERE natureza='acordao' AND tribunal=? AND ano=? AND relator LIKE ?",
        (m.group(1), int(m.group(2)), "%" + sobrenome + "%")).fetchone()[0]
    linhas.append({"trecho": t[:56], "candidatos": n})
inc = pd.DataFrame(linhas)
print(inc.candidatos.describe().round(1).to_string())
inc.head(20)

Das 35 `incompleta`, 27 trazem tribunal e ano suficientes para montar a consulta. A
mediana é de **6 candidatos** e o máximo chega a 21 — nunca 1. É a definição
operacional da classe: buscável, não resolvível. Um sistema que devolva o primeiro
candidato dessa lista está inventando um vínculo e, pela métrica, transformando
`incompleta` em `real`.

(Os casos com 0 candidatos são limitação da heurística de sobrenome usada aqui —
`t.split()[-1]` erra quando o nome do relator vem truncado por quebra de linha ou com
ruído de OCR, como `Assusete Magálhães`. Não significa que a citação resolva.)

## Impressões e recomendações

**O problema não é de modelagem, é de normalização e roteamento.** Um baseline sem
nenhum aprendizado de máquina — regex, reagrupamento de dígitos, correção de OCR
dentro do token, janela de cabeçalho por tribunal e uma regra de roteamento —
resolve 95% das citações reais de jurisprudência com τ de 2%, e 100% da classe `lei`.
O que sobra para um modelo é a extração dos spans e a classificação de forma, não a
decisão `real`/`inventada`, que é uma consulta.

**A ordem de prioridade que os dados sugerem:**

1. **Extração de spans.** É o gargalo real: sem o span, a citação não pontua, e o
   IoU ≥ 0,5 é o único requisito de borda. Um quarto dos trechos tem quebra de linha
   no meio, então o extrator precisa operar sobre texto corrido, não linha a linha.
2. **Roteamento por tipo antes de resolver.** Súmula, dispositivo de lei, acórdão e
   referência vaga têm caminhos de resolução diferentes e incompatíveis. Misturá-los
   é a maior fonte do erro caro.
3. **Normalização do identificador.** Reagrupar dígitos, tratar CNJ à parte, corrigir
   letra-por-dígito dentro do token. É aqui que mora a diferença entre os níveis 1 e 2,
   e o nível 2 vale o dobro.
4. **Só então a classificação**, que sai quase inteira da cardinalidade do resultado:
   1 candidato → `real`; 0 → `inventada`; 2+ ou nada para consultar → `incompleta`.

**Onde eu investiria o tempo que sobra:** no parser de cabeçalho da base, construído
uma vez offline. A base tem dois esquemas de identificação convivendo (STF/STJ com
`CLASSE Nº 1.741.784 - UF` no topo; TST/TSE/STM com CNJ) e 1.016 documentos. Indexar
`(classe, número, UF, tribunal) -> id` numa tabela própria elimina a varredura de FTS
por citação, elimina o problema de "quem cita versus quem é", e transforma a resolução
numa consulta por chave. É o conselho que a própria documentação dá, e os dados
confirmam que é o caminho certo.

**Sobre a fronteira da classe `incompleta`.** A revisão do gabarito de 04/09
(notebook 03) tirou dela as referências difusas: "jurisprudência pacífica desta
Corte" e afins não são citação. O critério que ficou é operacional — só é citação o
trecho que nomeia uma fonte concreta: tribunal, classe processual, relator, ano ou
estatuto. Trate o léxico dessas expressões difusas como **filtro negativo** no
extrator; cada uma que escapar é falso positivo direto na nota.